# monitor

> A folder somebody else is changing, looked at between turns, and the standing review that
> fires when it moves.

In [ ]:
#| default_exp monitor

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
from __future__ import annotations

import fnmatch, sys, threading, time, uuid
from collections import deque
from pathlib import Path

from fastcore.basics import patch
from fastcore.script import call_parse

from ramabana.core import AgentError, agent_err
from ramabana.tools import MAX_TOOL_CHARS, _diff, clip, delegate, err, summary
from shalya.core import one_line as _1

## Why a monitor

A watch in `vault` re-reads the world on a schedule: a page next week, a search next month. It
files what it finds and nothing looks at it until somebody asks. That is the wrong shape for the
case this module is for, which is a folder changing *now*, by something that is not this
conversation: a second agent working the same checkout, a build writing output, a colleague's
editor.

Three things have to hold for that to be useful, and none of them is a schedule.

- **A baseline, not a listing.** The first look is a baseline. Only what moves after it is a
change, so opening a watch on a repo does not produce a review of the whole repo.
- **The instructions are stored, not asked for again.** A change arrives when nobody is at the
keyboard. The brief the reviewer gets was written when the watch was opened, and it is all the
reviewer gets: it cannot see the conversation that opened it.
- **The review lands on the desk.** A review nobody reads is a wasted call. Each one is queued,
and the next turn's prompt carries it whether or not the model thought to ask.

## How a change is detected

By comparing snapshots, not by asking the operating system. `snapshot` is `{path: text}` for
every file the host's own `walk` will read under the folder; `changed` is the two-snapshot
difference. That needs no platform support, survives a process that was not watching when the
edit landed, and costs one walk per check.

The price is that a file too large to hold is tracked by its size alone, and that two changes
inside one settle window arrive as one review. Both are the right trade for the case: a burst of
edits by one agent *is* one change.

## What the review is

A read-only sub-agent, on the sub-agent model, with the session's read tools and `REVIEW_SP`. It
cannot edit, and that is deliberate rather than incidental: something else is editing this folder
while the review runs, and two writers on one tree is how work gets lost.

The snapshot advances whether the review succeeded or not. A model that keeps failing must not
turn one edit into a diff that grows for the rest of the session.

In [ ]:
#| export
#: What one look at a folder may hold. Past this the folder is being watched too widely, and
#: taking the first `SNAP_MAX_FILES` beats stalling a turn on a walk of somebody's home directory.
SNAP_MAX_FILES = 2000
SNAP_MAX_BYTES = 400_000     # per file. Past this a snapshot keeps the size, so a change still shows
REVIEW_MAX_CHARS = 24_000    # of change report one review prompt carries
REVIEW_MAX_STEPS = 12        # tool calls a reviewing sub-agent gets
PENDING_MAX = 20             # reviews held for the next turn before the oldest is dropped
DFLT_SETTLE = '20s'          # least time between two reviews of one folder


def secs(every):
    "Seconds from `'30s'`, `'5m'`, `'1h'`, or a number. One parser for the family, never our own."
    # pobblebonk first: it is where the family's schedules are moving, and it also takes `1h30m`
    try: from pobblebonk.core import secs as _secs
    except ImportError: from vishalakshi.acquire import secs as _secs
    return _secs(every)

In [ ]:
#| export
def _globs(pattern):
    "The globs in `pattern`, comma or space separated: `'*.py,*.ipynb'`."
    return [g for g in str(pattern or '').replace(',', ' ').split() if g]


def _matches(path, globs):
    "Whether `path` matches any of `globs`, by file name or by whole posix path."
    if not globs: return True
    return any(fnmatch.fnmatch(path.name, g) or fnmatch.fnmatch(path.as_posix(), g) for g in globs)


def files_under(host, folder, pattern=''):
    """Every readable file under `folder` matching `pattern`, sandbox-checked.

    The host's own `walk` decides what is readable, so a monitor skips exactly the generated
    directories and binary suffixes the rest of the tools skip. `check` refuses a folder outside
    the open roots, and it runs once rather than per file. Naming a single file watches that file.
    """
    root = host.check(folder, must_exist=True)
    globs = _globs(pattern)
    out = [p for p in (Path(x) for x in host.walk())
           if (p == root or root in p.parents) and _matches(p, globs)]
    return sorted(out)[:SNAP_MAX_FILES]


def snapshot(host, folder, pattern=''):
    "`{path: text}` for every watched file as it is now: one look, to compare against the next."
    out = {}
    for p in files_under(host, folder, pattern):
        t = host.text_at(p)
        if t is None: continue                     # unreadable this time; a later look may get it
        out[str(p)] = t if len(t) <= SNAP_MAX_BYTES else f'({len(t)} bytes, too large to diff)'
    return out


def changed(before, after):
    "`{path: (before, after)}` for every file added, edited or removed between two snapshots."
    out = {}
    for p, now in after.items():
        was = before.get(p)
        if was is None: out[p] = ('', now)
        elif was != now: out[p] = (was, now)
    for p, was in before.items():
        if p not in after: out[p] = (was, '')
    return out

In [ ]:
#| export
def _verb(was, now): return 'added' if not was else 'removed' if not now else 'edited'


def _counts(diff):
    "`(added, removed)` line counts from one unified diff."
    lines = diff.splitlines()
    add = sum(1 for l in lines if l.startswith('+') and not l.startswith('+++'))
    rem = sum(1 for l in lines if l.startswith('-') and not l.startswith('---'))
    return add, rem


def _rel(path, root=None):
    "`path` relative to the watched folder, or whole when it is not under one."
    p = Path(path)
    if root is None: return p.as_posix()
    try: rel = p.relative_to(root).as_posix()
    except ValueError: return p.as_posix()
    return p.name if rel == '.' else rel      # a watch on one file: the folder *is* the file


def summarise(changes):
    "One line: how many files were added, edited and removed."
    n = {}
    for was, now in changes.values(): n[_verb(was, now)] = n.get(_verb(was, now), 0) + 1
    return ', '.join(f'{v} {k}' for k, v in sorted(n.items())) or 'nothing'


def report(changes, folder='', mx=REVIEW_MAX_CHARS):
    """What changed: a line per file, then the unified diffs.

    The summary lines come first and are not clipped, so a review of a very large change still
    knows every file that moved even where it cannot see every diff.
    """
    root = Path(folder) if folder else None
    heads, diffs = [], []
    for p in sorted(changes):
        was, now = changes[p]
        rel = _rel(p, root)
        d = _diff(was, now, rel)
        add, rem = _counts(d)
        heads.append(f'{_verb(was, now):8} {rel}  +{add}/-{rem}')
        if d: diffs.append(d)
    head = '\n'.join(heads)
    if not diffs: return head
    room = max(0, mx - len(head) - 2)
    return head + '\n\n' + clip('\n\n'.join(diffs), room, more='read the files themselves')

In [ ]:
#| export
REVIEW_SP = """You are a review sub-agent inside a Python IDE. A folder the user is watching has \
changed, and you are being shown what changed together with the standing instructions they left \
for exactly this moment.

- Do what the instructions ask, about these changes, and nothing else.
- The diff is what moved. Read the surrounding files when the diff alone cannot answer the question.
- Report what you found, with paths and line numbers. Say so plainly when the change is fine: "no \
problem in these three files" is a useful answer and a manufactured concern is not.
- You cannot edit anything, and that is deliberate: something else is editing this folder right \
now. Describe the change you would make and stop.
- Nobody reads your working. Your answer is the only thing that reaches the user, and they cannot \
see the diff you were given, so name what you are talking about."""


def review_prompt(instructions, changes, folder=''):
    "The one self-contained question a reviewing sub-agent gets: the standing brief, then what moved."
    where = f' under {folder}' if folder else ''
    return f'{instructions}\n\nThese files changed{where}:\n\n{changes}'


def _attr(s):
    "One value, safe to sit inside a double-quoted tag attribute."
    return str(s).replace('"', "'")


def review_notice(recs, mx=REVIEW_MAX_CHARS):
    "The reviews that arrived since the last turn, as the block a prompt carries."
    if not recs: return ''
    out = [f'<folder-review folder="{_attr(r["folder"])}" files="{r["files"]}" status="{_attr(r["status"])}">\n'
           f'{r["summary"]}\n\n{r["review"] or r["error"] or r["changes"] or "(nothing)"}\n</folder-review>'
           for r in recs]
    return '\n\n' + clip('\n\n'.join(out), mx)

In [ ]:
#| export
class FolderWatch:
    "One monitored folder: what to look at, what to ask when it moves, and how it last looked."

    def __init__(self,
                 folder,               # the folder, or one file, to watch. Inside the open roots
                 instructions,         # the standing brief the reviewer gets, verbatim
                 pattern='',           # globs to limit it to. Empty watches everything readable
                 settle=DFLT_SETTLE,   # least time between two reviews, so one burst is one review
                 note=''):             # why it is being watched
        self.id = f'fw_{uuid.uuid4().hex[:8]}'
        self.folder, self.instructions = str(folder), str(instructions)
        self.pattern, self.note = str(pattern or ''), str(note or '')
        self.settle = secs(settle)
        self.snap = {}          # the last look. `Monitors.add` takes the first
        self.reviewed = None    # monotonic clock of the last review. The settle window runs from it
        self.reviews, self.last_status = 0, ''

    def __repr__(self): return f'FolderWatch({self.id} {self.folder} {len(self.snap)} files)'

In [ ]:
#| export
class Monitors:
    """Every folder this session watches, and the reviews they have produced.

    Session-scoped, which is the life of the thing it is for: another agent working the same
    checkout while this conversation is open. `check` is the tick, and it is safe to call from a
    background thread; `drain` is what the next turn's prompt takes.
    """

    def __init__(self,
                 host,
                 get_backend=None,   # callable -> the backend a review runs on, or None for no review
                 get_tools=None,     # callable -> the tools a reviewer may read the repo with
                 on_review=None):    # callable(record), for a frontend, per completed review
        self.host, self.get_backend, self.get_tools = host, get_backend, get_tools
        self.on_review = on_review
        self.watches = {}
        self.pending = deque(maxlen=PENDING_MAX)   # reviews no turn has carried yet
        self.lock = threading.Lock()               # guards `watches` and `pending`
        # one check at a time, whoever asked for it. The background tick and a `check_folders`
        # call otherwise both find the same change and pay for the same review twice
        self.checking = threading.Lock()

    def add(self, folder, instructions, pattern='', settle=DFLT_SETTLE, note=''):
        "Start watching `folder`. The first look is taken now, so only later changes are reviewed."
        if not str(instructions or '').strip():
            raise AgentError('a folder watch needs instructions: they are all the reviewer gets')
        w = FolderWatch(folder, instructions, pattern, settle, note)
        w.snap = snapshot(self.host, w.folder, w.pattern)
        with self.lock: self.watches[w.id] = w
        return w

    def remove(self, watch_id):
        "Stop watching one folder. Reviews already filed stay where they were filed."
        with self.lock: return self.watches.pop(str(watch_id), None) is not None

    def all(self):
        "Every watch, in the order they were opened."
        with self.lock: return list(self.watches.values())

    def drain(self):
        "Every review no turn has carried yet, oldest first, taken off the queue once."
        with self.lock:
            out = list(self.pending)
            self.pending.clear()
        return out

In [ ]:
#| export
@patch
def check(self: Monitors,
          force=False,   # look even inside a watch's settle window
          block=True     # wait for a check already running, rather than answering `None`
):
    """Look at every watched folder and review the ones that moved. One record per review.

    `None` rather than `[]` when another check holds the pass and `block` is off: "somebody is
    already reviewing this" and "nothing moved" are different answers to give a caller.
    """
    if not self.checking.acquire(blocking=block): return None
    try:
        out = []
        for w in self.all():
            try:
                if (rec := self._check(w, force=force)) is not None: out.append(rec)
            except Exception as e:
                w.last_status = 'error'
                out.append(self._record(w, 'error', error=agent_err(e)))
        return out
    finally: self.checking.release()


@patch
def _check(self: Monitors, w, force=False):
    "One folder: look, compare, and review when it moved. `None` when nothing did."
    now = time.monotonic()
    if not force and w.reviewed is not None and now - w.reviewed < w.settle: return None
    after = snapshot(self.host, w.folder, w.pattern)
    chg = changed(w.snap, after)
    if not chg: return None
    # the snapshot advances whether the review succeeds or not: a model that keeps failing must
    # not turn one edit into a diff that grows for the rest of the session
    w.snap, w.reviewed = after, now
    return self._review(w, chg)


@patch
def _review(self: Monitors, w, chg):
    "Run `w`'s standing instructions over what moved, as a read-only sub-agent."
    text = report(chg, w.folder)
    kw = dict(changes=text, files=len(chg), summary=summarise(chg))
    b = self.get_backend() if self.get_backend is not None else None
    if b is None: rec = self._record(w, 'unreviewed', **kw)
    else:
        tools = list(self.get_tools() or ()) if self.get_tools is not None else []
        answer = delegate(b, review_prompt(w.instructions, text, w.folder), tools,
                          sp=REVIEW_SP, max_steps=REVIEW_MAX_STEPS)
        rec = self._record(w, 'ok', review=answer, **kw)
    w.reviews += 1
    w.last_status = rec['status']
    return rec


@patch
def _record(self: Monitors, w, status, **kw):
    "One completed check: queued for the next turn, filed in memory, and returned."
    rec = dict(watch_id=w.id, folder=w.folder, status=status, when=time.time(), files=0,
               summary='', review='', changes='', error='') | dict(kw)
    with self.lock: self.pending.append(rec)
    self._file(rec)
    if self.on_review is not None:
        try: self.on_review(rec)
        except Exception: pass
    return rec


@patch
def _file(self: Monitors, rec):
    "Tell the user out of band, and put the review in durable memory when the host has any."
    line = f"{Path(rec['folder']).name}: {rec['summary'] or rec['status']}"
    try: self.host.note(f'folder review -- {line}')
    except Exception: pass
    if not rec['review']: return
    try: self.host.remember(rec['review'], title=f'folder review: {line}', tags=['folder-review'])
    except Exception: pass      # no memory on this host. The review still reaches the next turn

In [ ]:
#| export
def monitor_tools(get_monitors, mx=MAX_TOOL_CHARS):
    "Watching a folder somebody else is changing, and the review that fires when it moves."

    @summary(lambda a: f'Watch folder {_1(a.get("folder"), 80)}')
    def watch_folder(folder: str, instructions: str, pattern: str = '', settle: str = DFLT_SETTLE) -> str:
        """Watch `folder`, and review every later change to it against `instructions`.

        For work happening beside this conversation: another agent editing the repo, a build
        writing output, the user's own editor. Nothing fires for what is already there. The
        first look is taken now, and only what moves after it is reviewed.

        `instructions` is the standing brief, and it is all the reviewer gets: it cannot see this
        conversation. Write it self-contained -- "Review each change for correctness against the
        tests in tests/, and report anything that breaks a contract those tests assert."

        `pattern` limits the watch to matching files ('*.py', or '*.py,*.ipynb'). Empty watches
        everything readable. `settle` is the least time between two reviews, so one multi-file
        edit lands as one review rather than five.

        Reviews arrive on their own, in the turn after they finish. `check_folders` looks now.
        """
        try: w = get_monitors().add(folder, instructions, pattern=pattern, settle=settle)
        except Exception as e: return err('could not watch that folder', e)
        which = f' matching {w.pattern}' if w.pattern else ''
        return f'watching {w.folder} as {w.id} ({len(w.snap)} files{which}); every change is reviewed'

    @summary(lambda a: 'List watched folders')
    def list_folder_watches() -> str:
        "Every folder being watched, what it is reviewed for, and how many reviews it has produced."
        ws = get_monitors().all()
        if not ws: return 'no folder is being watched'
        return clip('\n'.join(
            f"{w.id}  {w.folder}  {len(w.snap)} files  settle={int(w.settle)}s  reviews={w.reviews}"
            f"  {w.last_status or 'never fired'}  {w.instructions[:80]}" for w in ws), mx)

    @summary(lambda a: f'Stop watching {a.get("watch_id","?")}')
    def cancel_folder_watch(watch_id: str) -> str:
        "Stop watching one folder. Only when the user asks. Reviews already filed stay in memory."
        if get_monitors().remove(watch_id): return f'stopped watching {watch_id}'
        return f'no such folder watch: {watch_id}'

    @summary(lambda a: 'Check watched folders')
    def check_folders() -> str:
        """Look at every watched folder now, and report every review that is waiting.

        Ignores the settle window, so this is the tool for "what has that other agent just done".
        Each review is reported once: what this returns will not arrive again on a later turn.
        A review already running is not waited for; its answer arrives on your next turn.
        """
        m = get_monitors()
        if not m.all(): return 'no folder is being watched'
        try: busy = m.check(force=True, block=False) is None
        except Exception as e: return err('could not check the watched folders', e)
        recs = m.drain()
        if not recs:
            if busy: return 'a review is already running; its answer arrives on your next turn'
            return 'nothing has changed since the last look'
        return clip('\n\n'.join(
            f"{r['folder']} -- {r['summary'] or r['status']}\n"
            f"{r['review'] or r['error'] or r['changes']}" for r in recs), mx)

    return [watch_folder, list_folder_watches, cancel_folder_watch, check_folders]

In [ ]:
from ramabana.testing import FakeBackend, MemHost, SPEC

In [ ]:
h = MemHost({'/proj/a.py': 'def a(): return 1\n', '/proj/notes.md': 'hello\n'})
m = Monitors(h, get_backend=lambda: FakeBackend(SPEC))
w = m.add('/proj', 'Review each change for correctness.')
test_eq(len(w.snap), 2)          # the first look is a baseline, not a change
test_eq(m.check(), [])

h.write('/proj/a.py', 'def a(): return 2\n')
rec, = m.check()
test_eq((rec['status'], rec['files'], rec['summary']), ('ok', 1, '1 edited'))
test_eq(rec['review'], 'sub answer')          # what `FakeBackend.spawn` replies
assert '-def a(): return 1' in rec['changes'] and '+def a(): return 2' in rec['changes']

In [ ]:
# a burst inside the settle window is one change, and `force` is how a tool call looks anyway
h.write('/proj/b.py', 'def b(): pass\n')
test_eq(m.check(), [])
rec, = m.check(force=True)
test_eq((rec['summary'], rec['status']), ('1 added', 'ok'))

del h.files['/proj/notes.md']
test_eq(m.check(force=True)[0]['summary'], '1 removed')

In [ ]:
# every review reaches the next turn once, and `drain` is the only way it does
test_eq(len(m.drain()), 3)
test_eq(m.drain(), [])
test_eq(review_notice([]), '')

test_eq(_globs('*.py, *.ipynb'), ['*.py', '*.ipynb'])
test_eq(summarise({'a': ('', 'x'), 'b': ('x', 'y')}), '1 added, 1 edited')
test_fail(lambda: m.add('/proj', '  '), contains='needs instructions')

In [ ]:
# one change is reviewed once: a background tick and a `check_folders` call must not both pay
h.write('/proj/a.py', 'def a(): return 3\n')
m.checking.acquire()
try: test_eq(m.check(block=False), None)          # `None`, not `[]`: somebody else is looking
finally: m.checking.release()
test_eq(m.check(force=True)[0]['summary'], '1 edited')

## The beat

`Monitors` looks at a folder between turns, so nothing it watches moves while no turn is running.
The clock that fixes that is not Ramabana's. pobblebonk schedules an operating-system job through
cron, launchd or `schtasks`, and holds the schedules, the fire queue and a notes stream in one
SQLite database.

So the two halves never call each other. The beat writes notes and a session reads them, each
under its own reader id, and `drain` is what remembers how far a reader got.

What Ramabana keeps here is only the seam: how to open that database, how to read what the beat
left, and an entry point a beat can run. The schedules themselves belong to whichever package owns
the work. vishalakshi owns the watches.

In [ ]:
#| export
POB_READER = 'ramabana'   #: the notes-stream reader a session drains under
TICKS = {}                #: schedule name -> what a beat runs for it. `ramabana-tick` registers these

def on_tick(name):
    "Register the callback a fire of `name` runs, for whichever process the beat starts."
    def _f(fn):
        TICKS[str(name)] = fn
        return fn
    return _f

def _pob_home():
    "Where the beat keeps its launcher, its log and the database. pobblebonk's answer, or its default."
    try: from pobblebonk.heartbeat import HOME
    except ImportError: return Path.home()/'.pobblebonk'
    return Path(HOME)

#: One knob, read once. Both halves ask `pob_path`, so neither can open a file the other does not
POB_HOME = _pob_home()

def pob_path(): return Path(POB_HOME).expanduser()/'pob.db'

def pob(path=None):
    "The database the beat and this session share. None when pobblebonk is not installed."
    # `Pob(None)` makes a temporary database the beat would never find again, so always name one
    try: from pobblebonk.core import Pob
    except ImportError: return None
    return Pob(str(path or pob_path()))

def beat_notes(db, reader=POB_READER, limit=20):
    "What the beat left that `reader` has not read. Empty for no beat, and never raises into a turn."
    if db is None: return []
    try: got = db.drain(str(reader), limit)
    except Exception: return []
    return [f"{n['title']}: {n['body']}" if n.get('body') else str(n['title']) for n in got]

def beat_notice(notes, mx=REVIEW_MAX_CHARS):
    "What the beat left since this session last looked, as the block a prompt carries."
    if not notes: return ''
    return '\n\n<beat>\n' + clip('\n'.join(notes), mx) + '\n</beat>'


`ramabana-tick` is what a beat runs. It registers whatever `on_tick` collected, ticks once, and
leaves. A cold process with no schedules registered is not an error: the schedules belong to the
packages that own the work, and a machine may have none of them installed yet.

In [ ]:
#| export
BEAT_TAG = 'ramabana'   #: names this package's launcher, its log and the scheduled job

def heartbeat():
    "pobblebonk's scheduler seam, or None when it is not installed."
    try: from pobblebonk import heartbeat
    except ImportError: return None
    return heartbeat

@call_parse
def tick(db: str = '',         # the shared database; pobblebonk's own by default
         quiet: bool = False,  # say nothing on success
         install: bool = False,# schedule this command instead of running one beat
         uninstall: bool = False,  # stop the scheduled beat
         every: int = 60):     # seconds between beats, when installing. Whole minutes
    "One beat: run the schedules that are due and leave what they found as notes."
    if install or uninstall:
        hb = heartbeat()
        if hb is None:
            print("scheduling needs pobblebonk: pip install 'ramabana[cron]'", file=sys.stderr)
            return 1
        if uninstall:
            print('stopped' if hb.uninstall(BEAT_TAG) else 'nothing was scheduled')
            return 0
        cmd = f'{Path(sys.executable).parent/"ramabana-tick"} --quiet'
        print(f'every {every}s: {hb.install(cmd, tag=BEAT_TAG, every=every)}')
        return 0
    p = pob(db or None)
    if p is None:
        print("pobblebonk is not installed: pip install 'ramabana[cron]'", file=sys.stderr)
        return 1
    for name, fn in TICKS.items(): p.on(name)(fn)
    got = p.tick()
    if not quiet: print(f'{len(got.ran)} ran of {len(got.fired)} fired')
    return 0

In [ ]:
# one beat, end to end: a schedule fires, its callback runs, and the note reaches a session
import contextlib, io, tempfile, time

@on_tick('demo')
def _demo(fire): return 'the beat ran this'

db = str(Path(tempfile.mkdtemp())/'pob.db')
pob(db).add('demo', every='1s')
time.sleep(1.2)
out = io.StringIO()
with contextlib.redirect_stdout(out): test_eq(tick.__wrapped__(db=db), 0)
test_eq(out.getvalue().strip(), '1 ran of 1 fired')
test_eq(beat_notes(pob(db)), ['demo: the beat ran this'])
del TICKS['demo']

In [ ]:
import tempfile
beat = pob(Path(tempfile.mkdtemp())/'pob.db')     # what pobblebonk gives a real machine
beat.note('watches', '2 of 5 fired')
beat.note('nightly', '')
beat_notes(beat)

In [ ]:
test_eq(beat_notes(beat), [])                      # a reader gets each note once
test_eq(beat_notes(beat, reader='another'), ['watches: 2 of 5 fired', 'nightly'])
test_eq(beat_notes(None), [])                     # pobblebonk not installed is not an error

class _Broken:
    def drain(self, reader, limit=20): raise RuntimeError('the database is locked')
test_eq(beat_notes(_Broken()), [])                # a broken beat never takes the turn down with it

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()